# Discover Page 登録内容：採算KPIの定義

> Discover Pages は Beta 機能です。画面の項目名・配置は変わる可能性があります。
> 機密原価・個人情報・規制対象データは記載しません。KPI の**定義**だけを書き、**数値**は Metric View から取得します。

## 登録フィールド

※ テーブル名は Free Edition の既定値（`workspace.vehicle_alias_handson`）です。`config/00_config` で変更した場合は、自分のカタログ・スキーマ名に読み替えてください。

| 項目 | 入力値 |
|---|---|
| Domain | `Vehicle Profitability` |
| Page名 | `採算KPIの定義` |
| Description | 経営ダッシュボードと Genie で使う車種別採算KPI（売上・材料費・1台あたり簡易限界利益・データ信頼度）の正式な定義 |
| Synonyms | `採算KPI` / `経営KPI` / `1台あたり利益` / `限界利益` / `材料費差額` / `計画比` / `データ信頼度` / `変換率` |
| Related assets | `workspace.vehicle_alias_handson.mv_vehicle_profitability`<br>`workspace.vehicle_alias_handson.v_data_trust_summary`<br>`workspace.vehicle_alias_handson.v_exec_action_items`<br>`workspace.vehicle_alias_handson.v_vehicle_monthly_profitability`<br>ダッシュボード「車種別採算 経営サマリ（ハンズオン）」 |
| Sources | 「車種別採算KPI 定義書 KPI-VP-001 Rev.1（架空）」<br>`workspace.vehicle_alias_handson.mv_vehicle_profitability`（KPI の計算式の正本） |

---

## 本文（以下をそのまま Page 本文に貼り付け）

### 目的

経営者が「この車種は計画どおり儲かっているか」「どこに手を打つべきか」を判断するための KPI を定義します。KPI の**計算式の正本は Metric View `mv_vehicle_profitability`** です。ダッシュボード・Genie・SQL のどれで見ても、同じ定義の数字が出ます。

### KPI の一覧

| KPI | 定義 | 単位 | 見方 |
|---|---|---|---|
| 実績売上 | 販売実績の売上合計 | 百万円 | — |
| 売上計画比 | 実績売上 ÷ 計画売上 − 1 | % | マイナスは計画未達 |
| 材料費差額 | 実績材料費 − 計画材料費 | 百万円 | **プラスは計画超過**（悪化） |
| 1台あたり実績売上 | 実績売上 ÷ 販売台数 | 百万円/台 | — |
| 1台あたり実績材料費 | 実績材料費 ÷ **生産台数** | 百万円/台 | 材料費は生産側で発生するため生産台数で割る |
| 1台あたり簡易限界利益 | 1台あたり売上 − 1台あたり材料費 | 百万円/台 | 計画と比べて、1台の稼ぐ力が落ちていないかを見る |
| データ信頼度（売上の変換率） | 共通機種IDに変換でき、集計に入っている売上 ÷ 全売上 | % | 100% に近いほど、数字の抜け漏れが少ない |

### 集計のルール

- 対象は、共通機種IDに変換できたデータ（`resolution_status = 'RESOLVED'`）だけです。変換できないデータは集計に含めず、「集計に入っていない売上」として別に表示します。
- 1台あたりの KPI は、**合計どうしを割って**計算します。月別の値を平均したものではありません（Metric View が自動でこの計算をします）。
- 「1台あたり簡易限界利益」は、ハンズオン用の簡易指標です。加工費・物流費・販売費などは含みません。正式な利益指標と混同しないでください。
- 金額の単位は百万円です（ハンズオン用の架空データ）。

### データ信頼度の判定

| 判定 | 条件 | 経営者への伝え方 |
|---|---|---|
| 良好 | 重要な品質ルール違反がなく、売上・材料費の変換率がどちらも 99% 以上 | そのまま判断に使える |
| 要注意（未変換のデータあり） | 変換率が 99% 未満 | 集計に入っていない金額を確認してから判断する |
| 要注意（重要な品質ルール違反あり） | タイヤ本数・部品の重複・マスタの矛盾（DQ-1〜3）の違反が 1 件以上 | 材料費などが実態とずれている可能性がある。打ち手の一覧で原因を確認する |

### 回答するときの注意

- KPI の数字を答えるときは、**データ信頼度の判定もあわせて伝えてください**。
- 「儲かっているか」と聞かれた場合は、売上計画比と 1台あたり簡易限界利益（計画との比較）の両方を示してください。
- 「どこに手を打つべきか」と聞かれた場合は、`v_exec_action_items`（打ち手の一覧）の優先順に答えてください。
- 材料費の計画超過が見つかった月は、部品表（BOM）の品質ルール違反（DQ-1・DQ-2）とあわせて確認するよう促してください。

### 管理

- 定義のオーナー：経営企画（架空）
- 変更手順：定義書を改訂し、`mv_vehicle_profitability` を更新してから、このPageを更新する